## Sales

In [0]:
from pyspark.sql import functions as F

source_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/landing_sales"
) 

schema_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_schemas/landing_sales"
)

checkpoint_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_checkpoints/landing_sales"
)

target_table = (
    "data_lakehouse_databricks."
    "bronze.bronze_crm_sales_details"
)

# Get exactly the columns expected by the existing Bronze table
target_columns = spark.table(target_table).columns

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)

    # Removes _rescued_data and any other Auto Loader metadata
    .select(*target_columns)
)

query = (
    df_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
# Display the number of rows processed by Auto Loader
print(f"Auto Loader processed {query.lastProgress['numInputRows']} rows in the last batch")

## Customers

In [0]:
source_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/landing_cust"
)

schema_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_schemas/landing_cust"
)

checkpoint_path = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_checkpoints/landing_cust"
)

target_table = (
    "data_lakehouse_databricks."
    "bronze.bronze_crm_cust_info"
)


target_columns = spark.table(target_table).columns

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
    .select(*target_columns)
)

query = (
    df_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*)
FROM data_lakehouse_databricks.bronze.bronze_crm_sales_details;